# Retrieval Query Strategy Comparison (Qdrant)

This notebook evaluates three retrieval query strategies over 10 random samples from `evaluation.json` using Qdrant at `localhost:6333`.

Strategies:
- Regex/string-based (helper functions) intelligent query generation.

In [ ]:
import json
import random
import re
from pathlib import Path


import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client import models

RANDOM_SEED = 42
SAMPLE_SIZE = 10
TOP_K = 10
COLLECTION_NAME = "guideline_embeddings"
MODEL_NAME = "BAAI/bge-large-en-v1.5"

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
EVAL_PATH = DATA_DIR / "evaluation.json"
EVAL_FILES_DIR = DATA_DIR / "evaluation_files"

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation file: {EVAL_PATH}")

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project
Evaluation file: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\evaluation.json


In [9]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_processing.indent_signal import check_indent_errors

In [2]:
# Load evaluation data and select 10 random samples
with EVAL_PATH.open("r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

random.seed(RANDOM_SEED)
sampled_entries = random.sample(evaluation_data, k=min(SAMPLE_SIZE, len(evaluation_data)))

def resolve_source_file(entry):
    source_path = Path(entry["source_file"])
    if source_path.is_absolute():
        return source_path
    return EVAL_FILES_DIR / source_path.name if not (EVAL_FILES_DIR / source_path).exists() else EVAL_FILES_DIR / source_path

for entry in sampled_entries:
    file_path = resolve_source_file(entry)
    entry["resolved_source_file"] = str(file_path)
    entry["file_text"] = file_path.read_text(encoding="utf-8", errors="ignore")

print(f"Loaded {len(evaluation_data)} total entries")
print(f"Sampled {len(sampled_entries)} entries")
pd.DataFrame([
    {
        "id": e["id"],
        "repo": e["repo"],
        "source_path": e["source_path"],
        "resolved_source_file": e["resolved_source_file"]
    }
    for e in sampled_entries
])

Loaded 97 total entries
Sampled 10 entries


,id,repo,source_path,resolved_source_file
0,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
1,synthetic-django_PR_36,kannan-dedsec/synthetic-django,urls.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
2,synthetic-django_PR_24,kannan-dedsec/synthetic-django,managers.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
3,synthetic-sklearn_PR_38,kannan-dedsec/synthetic-sklearn,tests/test_models.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
4,synthetic-fastapi_PR_38,kannan-dedsec/synthetic-fastapi,tests/test_users.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
5,synthetic-fastapi_PR_34,kannan-dedsec/synthetic-fastapi,schemas/item.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
6,synthetic-fastapi_PR_31,kannan-dedsec/synthetic-fastapi,routers/auth.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
7,synthetic-django_PR_39,kannan-dedsec/synthetic-django,views.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
8,synthetic-django_PR_35,kannan-dedsec/synthetic-django,tests/test_views.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...
9,synthetic-sklearn_PR_30,kannan-dedsec/synthetic-sklearn,hyperparameter_tuning.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...


In [3]:
# Connect to Qdrant and load embedding model
client = QdrantClient(url="http://localhost:6333")
collection_info = client.get_collection(COLLECTION_NAME)
print(f"Connected to Qdrant. Collection: {COLLECTION_NAME}")
print(f"Points count: {collection_info.points_count}")

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")

Connected to Qdrant. Collection: guideline_embeddings
Points count: 505


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2681.83it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: BAAI/bge-large-en-v1.5


In [4]:
COMMON_SOURCE_TYPES = ["pep8", "flake8", "pylint", "ruff"]

def repo_family(repo_name):
    repo_lower = repo_name.lower()
    for family in ["django", "fastapi", "flask", "pandas", "sklearn", "scikit-learn"]:
        if family in repo_lower:
            return "scikit-learn" if family == "sklearn" else family
    return None

def build_query_filter(repo_name):
    family = repo_family(repo_name)
    should_conditions = [
        models.FieldCondition(key="source_type", match=models.MatchValue(value=s))
        for s in COMMON_SOURCE_TYPES
    ]
    if family:
        should_conditions.insert(
            0,
            models.FieldCondition(
                key="source_type",
                match=models.MatchValue(value=f"{family}_guidelines"),
            ),
        )
        should_conditions.insert(
            1,
            models.FieldCondition(
                key="source_type",
                match=models.MatchValue(value=f"{family}_review_comment"),
            ),
        )
    return models.Filter(should=should_conditions)

def retrieve_guidelines(query_text, repo_name, top_k=TOP_K):
    query_vector = model.encode(query_text).tolist()
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=build_query_filter(repo_name),
        limit=top_k,
    )
    return response.points

print("Retrieval helpers ready")

Retrieval helpers ready


In [10]:
# Strategy 2: Regex/string-based intelligent query
def query_strategy_regex_intelligent(entry):
    text = entry["file_text"]
    lines = text.splitlines()

    imports = [ln.strip() for ln in lines if re.match(r"^\s*(import|from)\s+", ln)]
    camel_case_identifiers = set(re.findall(r"\b[a-z]+[A-Z][A-Za-z0-9_]*\b", text))
    mutable_defaults = re.findall(r"def\s+\w+\([^)]*=\s*(\[\]|\{\}|dict\(|list\()", text)

    # Use the check_indent_errors function from indent_signal.py
    indent_issues = check_indent_errors(entry["resolved_source_file"])

    docstring_markers = len(re.findall(r"'''|\"\"\"", text))

    signals = [
        f"repo family: {repo_family(entry['repo'])}",
        f"import statements count: {len(imports)}",
        f"camelCase identifiers: {', '.join(sorted(list(camel_case_identifiers))[:8]) or 'none'}",
        f"mutable-default patterns: {len(mutable_defaults)}",
        f"indentation issues: {len(indent_issues)}",
        f"docstring markers count: {docstring_markers}",
        "focus on naming_convention, unused_import, indentation, mutable_default, documentation_formatting"
    ]

    return " ; ".join(signals)

print("Strategy 2 query builders ready")

Strategy 2 query builders ready


In [11]:
# Run strategy 2 over sampled entries and collect retrieved chunks with timing
import time

chunk_rows = []
for entry in sampled_entries:
    try:
        query_text = query_strategy_regex_intelligent(entry)
    except Exception as e:
        query_text = f"<error building query: {e}>"

    start = time.perf_counter()
    try:
        points = retrieve_guidelines(query_text, entry.get('repo', ''), top_k=TOP_K)
    except Exception as e:
        points = []
    end = time.perf_counter()

    retrieval_time = end - start

    # Safely extract payload representations for storage
    payloads = []
    for p in points:
        try:
            payload = getattr(p, 'payload', p)
        except Exception:
            payload = p
        payloads.append(str(payload))

    if payloads:
        per_chunk_time = retrieval_time / max(1, len(payloads))
        for idx, payload in enumerate(payloads):
            chunk_rows.append({
                'id': entry.get('id'),
                'repo': entry.get('repo'),
                'source_path': entry.get('source_path'),
                'resolved_source_file': entry.get('resolved_source_file'),
                'query': query_text,
                'retrieved_count': len(payloads),
                'chunk_index': idx,
                'retrieved_payload': payload,
                'retrieval_time': retrieval_time,
                'time_taken': per_chunk_time,
            })
    else:
        # No points returned; record a single row with empty payload and the retrieval time
        chunk_rows.append({
            'id': entry.get('id'),
            'repo': entry.get('repo'),
            'source_path': entry.get('source_path'),
            'resolved_source_file': entry.get('resolved_source_file'),
            'query': query_text,
            'retrieved_count': 0,
            'chunk_index': None,
            'retrieved_payload': None,
            'retrieval_time': retrieval_time,
            'time_taken': retrieval_time,
        })

results_df = pd.DataFrame(chunk_rows)
results_df.head()

,id,repo,source_path,resolved_source_file,query,retrieved_count,chunk_index,retrieved_payload,retrieval_time,time_taken
0,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...,repo family: scikit-learn ; import statements ...,10,0,{'text': 'N813: CamelCase imported as lowercas...,7.629858,0.762986
1,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...,repo family: scikit-learn ; import statements ...,10,1,{'text': 'N817: CamelCase imported as acronym....,7.629858,0.762986
2,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...,repo family: scikit-learn ; import statements ...,10,2,{'text': 'N814: CamelCase imported as constant...,7.629858,0.762986
3,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...,repo family: scikit-learn ; import statements ...,10,3,"{'text': 'Python packages should have short, a...",7.629858,0.762986
4,synthetic-sklearn_PR_24,kannan-dedsec/synthetic-sklearn,custom_transformer.py,C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1...,repo family: scikit-learn ; import statements ...,10,4,{'text': 'W0404 (reimported): A module is impo...,7.629858,0.762986


In [24]:
# Aggregate results by PR and show ground-truth violations with retrieved chunks
# Load ground_truth_reviews directly from evaluation.json
with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    full_eval = json.load(f)

print(f"Loaded full evaluation data for ground truth mapping: {len(full_eval)} entries")
for i in range(3):
    print(f"Sample entry {i}: ID={full_eval[i].get('id')}, GT Violations={full_eval[i].get('ground_truth_reviews')}")
gt_map = {e.get('id'): e.get('ground_truth_reviews') for e in full_eval}

Loaded full evaluation data for ground truth mapping: 97 entries
Sample entry 0: ID=synthetic-django_PR_21, GT Violations=[{'line_number': 9, 'violation_category': 'unused_import', 'review_comment': 'Unused import: os'}, {'line_number': 10, 'violation_category': 'unused_import', 'review_comment': 'Unused import: sys'}, {'line_number': 11, 'violation_category': 'unused_import', 'review_comment': 'Unused import: re'}]
Sample entry 1: ID=synthetic-django_PR_22, GT Violations=[{'line_number': 43, 'violation_category': 'naming_convention', 'review_comment': 'camelCase function name: cleanTitle'}, {'line_number': 76, 'violation_category': 'naming_convention', 'review_comment': 'camelCase function name: cleanName'}]
Sample entry 2: ID=synthetic-django_PR_23, GT Violations=[{'line_number': 11, 'violation_category': 'unused_import', 'review_comment': 'Unused import: os'}, {'line_number': 12, 'violation_category': 'unused_import', 'review_comment': 'Unused import: sys'}, {'line_number': 13, 'vio

In [27]:
# temp save for results_df.to_csv("retrieval_strategy_2_results.csv", index=False)
results_df.to_csv("retrieval_strategy_2_results.csv", index=False)

In [25]:
gt_map

{'synthetic-django_PR_21': [{'line_number': 9,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: os'},
  {'line_number': 10,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: sys'},
  {'line_number': 11,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: re'}],
 'synthetic-django_PR_22': [{'line_number': 43,
   'violation_category': 'naming_convention',
   'review_comment': 'camelCase function name: cleanTitle'},
  {'line_number': 76,
   'violation_category': 'naming_convention',
   'review_comment': 'camelCase function name: cleanName'}],
 'synthetic-django_PR_23': [{'line_number': 11,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: os'},
  {'line_number': 12,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: sys'},
  {'line_number': 13,
   'violation_category': 'unused_import',
   'review_comment': 'Unused import: re'},
  {'line_number': 24

In [28]:
# Group retrieved chunks per PR id and print ground-truth and retrieved chunks
# Load ground-truth list from evaluation.json if available
try:
    with open(EVAL_PATH, 'r', encoding='utf-8') as f:
        full_eval = json.load(f)
    gt_map_list = {e.get('id'): e.get('ground_truth_reviews') for e in full_eval}
except Exception:
    gt_map_list = {e.get('id'): e.get('ground_truth_reviews') for e in sampled_entries}

# Build grouped metadata for iteration
grouped = (
    results_df.groupby('id')
    .agg(
        repo=('repo', 'first'),
        resolved_source_file=('resolved_source_file', 'first'),
        retrieved_count=('retrieved_count', 'first')
    )
    .reset_index()
)

# Iterate and print as requested
for _, row in grouped.iterrows():
    pid = row['id']
    print(f"ID: {pid} | repo: {row['repo']} | resolved: {row['resolved_source_file']}")

    # Ground-truth reviews
    gt_list = gt_map_list.get(pid)
    if gt_list:
        for gt in gt_list:
            ln = gt.get('line_number') if isinstance(gt, dict) else None
            cat = gt.get('violation_category') if isinstance(gt, dict) else None
            comment = gt.get('review_comment') if isinstance(gt, dict) else str(gt)
            print(f"  GT: ({ln}, {cat}, {comment})")
    else:
        print("  GT: None")

    # Retrieved chunks for this PR
    rows_for_id = results_df[results_df['id'] == pid]
    if not rows_for_id.empty:
        for _, r in rows_for_id.iterrows():
            idx = r.get('chunk_index')
            tt = r.get('time_taken')
            payload = r.get('retrieved_payload')
            print(f"  Retrieved: (chunk_index={idx}, time_taken={tt}, payload={payload})")
    else:
        print("  Retrieved: None")

    print("---")

ID: synthetic-django_PR_24 | repo: kannan-dedsec/synthetic-django | resolved: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\evaluation_files\synthetic-django_PR_24_managers.py
  GT: (32, naming_convention, camelCase function name: byAuthor)
  GT: (42, naming_convention, camelCase function name: withinDateRange)
  GT: (83, naming_convention, camelCase function name: byAuthor)
  GT: (93, naming_convention, camelCase function name: withinDateRange)
  GT: (32, mutable_default, Mutable default [] for param 'author_id')
  GT: (42, mutable_default, Mutable default [] for param 'start_date')
  GT: (42, mutable_default, Mutable default {} for param 'end_date')
  GT: (83, mutable_default, Mutable default [] for param 'author_id')
  GT: (93, mutable_default, Mutable default [] for param 'start_date')
  GT: (93, mutable_default, Mutable default {} for param 'end_date')
  Retrieved: (chunk_index=0, time_taken=0.5805328100046608, payload={'text': "The indentatio

In [12]:
results_df["query"].value_counts()

query
repo family: scikit-learn ; import statements count: 4 ; camelCase identifiers: filteredData, fitTransform, lowerBound, upperBound ; mutable-default patterns: 2 ; indentation issues: 0 ; docstring markers count: 12 ; focus on naming_convention, unused_import, indentation, mutable_default, documentation_formatting                                       10
repo family: django ; import statements count: 2 ; camelCase identifiers: urlPatterns ; mutable-default patterns: 0 ; indentation issues: 6 ; docstring markers count: 2 ; focus on naming_convention, unused_import, indentation, mutable_default, documentation_formatting                                                                                     10
repo family: django ; import statements count: 2 ; camelCase identifiers: byAuthor, withinDateRange ; mutable-default patterns: 4 ; indentation issues: 0 ; docstring markers count: 24 ; focus on naming_convention, unused_import, indentation, mutable_default, documentation_formattin

In [ ]:
results_df